In [2]:
YOUR_KEY_ID = None # id статического ключа
YOUR_SECRET_KEY = None # секрет статического ключа
BUCKET_NAME = None # название бакета (без s3://)
ENDPOINT_URL = None # эндпоинт
REGION = None # регион

In [4]:
import os
import tqdm
import torch
import boto3
import random
import numpy as np
import polars as pl
import pandas as pd




from botocore.client import Config
from datasets import load_dataset
from pathlib import Path
from io import BytesIO


In [3]:
session = boto3.session.Session()
s3_client = session.client(
    's3',
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=YOUR_KEY_ID,
    aws_secret_access_key=YOUR_SECRET_KEY,
    config=Config(region_name="ru-central1-a")
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "seed": SEED,
    "history_length": 50,
    "num_candidates": 4,
    "vocab_size": 32,
    "message_length": 4,
    "batch_size": 1024,
    "embedding_dim": 64,
    "hidden_dim": 128,
}

print("Polars:", pl.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [6]:
pools = ["train", "test", "val", "test_ranking", "val_ranking", "likes_train",
    "likes_val",
    "likes_test",
    "dislikes_train",
    "dislikes_val",
    "dislikes_test",
    "listens_train",
    "listens_val",
    "listens_test"]
s3_paths = {
    name: f"s3://{BUCKET_NAME}/data/yambda/{name}.parquet"
    for name in pools
}

In [7]:
scanning = lambda x: pl.scan_parquet(
    s3_paths[x],
    storage_options = {
        "aws_access_key_id": YOUR_KEY_ID,
        "aws_secret_access_key": YOUR_SECRET_KEY,
        "aws_region": "ru-central1-a",
        "endpoint_url": ENDPOINT_URL
    }
)

In [8]:
train = scanning("train")
test = scanning("test")
val = scanning("val")
test_ranking = scanning("test_ranking")
val_ranking = scanning("val_ranking")

In [9]:
likes_train = scanning("likes_train")
likes_val = scanning("likes_val")
likes_test = scanning("likes_test")

dislikes_train = scanning("dislikes_train")
dislikes_val = scanning("dislikes_val")
dislikes_test = scanning("dislikes_test")

listens_train = scanning("listens_train")
listens_val = scanning("listens_val")
listens_test = scanning("listens_test")

In [10]:
import polars as pl

EVENT_TYPES = {
    "like": 1,
    "dislike": 2,
    "listen": 3,
}

EVENT_SCHEMA = {
    "uid": pl.UInt32,
    "item_id": pl.UInt32,
    "timestamp": pl.UInt32,
    "event_type": pl.UInt8,
    "is_organic": pl.UInt8,
    "played_ratio_pct": pl.UInt16,
    "track_length_seconds": pl.UInt32,
}


def prepare_events(lf: pl.LazyFrame, event_type: str):
    """
    Приводит события одного типа к общей схеме.
    Не выполняет collect().
    """
    assert event_type in EVENT_TYPES

    is_listen = event_type == "listen"

    return lf.select(
        pl.col("uid"),
        pl.col("item_id"),
        pl.col("timestamp"),

        pl.lit(EVENT_TYPES[event_type])
          .cast(pl.UInt8)
          .alias("event_type"),

        pl.col("is_organic"),

        (
            pl.col("played_ratio_pct")
            if is_listen
            else pl.lit(None).cast(pl.UInt16)
        ).alias("played_ratio_pct"),

        (
            pl.col("track_length_seconds")
            if is_listen
            else pl.lit(None).cast(pl.UInt32)
        ).alias("track_length_seconds"),
    )


def build_event_stream(split):
    """Объединяет три типа событий одного сплита."""

    return pl.concat(
        [
            prepare_events(
                globals()[f"likes_{split}"], "like"
            ),
            prepare_events(
                globals()[f"dislikes_{split}"], "dislike"
            ),
            prepare_events(
                globals()[f"listens_{split}"], "listen"
            ),
        ],
        how="vertical",
    )

In [11]:
events_train = build_event_stream("train")
events_val = build_event_stream("val")
events_test = build_event_stream("test")

for name, lf in [
    ("train", events_train),
    ("val", events_val),
    ("test", events_test),
]:
    print(f"\n{name}")
    print(lf.collect_schema())
    display(lf.head(5).collect())


train
Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])


uid,item_id,timestamp,event_type,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u8,u16,u32
100,732449,44755,1,1,null,null
100,6568592,1155860,1,0,null,null
100,5411243,1259125,1,1,null,null
100,7371186,1260005,1,0,null,null
100,4943655,1263935,1,0,null,null



val
Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])


uid,item_id,timestamp,event_type,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u8,u16,u32
300,9286415,25907210,1,1,null,null
400,5001732,25900470,1,1,null,null
1100,5905771,25893380,1,0,null,null
1100,9014511,25895020,1,0,null,null
1600,2612724,25896555,1,0,null,null



test
Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])


uid,item_id,timestamp,event_type,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u8,u16,u32
700,3383415,25939545,1,0,null,null
1400,7707460,25993865,1,1,null,null
1400,6005845,25993945,1,1,null,null
1400,7539311,25993945,1,0,null,null
1400,8908728,25993975,1,1,null,null


In [12]:
events_train = build_event_stream("train")
events_val = build_event_stream("val")
events_test = build_event_stream("test")

for name, lf in [
    ("train", events_train),
    ("val", events_val),
    ("test", events_test),
]:
    print(f"\n{name}")
    print(lf.collect_schema())
    display(lf.head(5).collect())


train
Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])


uid,item_id,timestamp,event_type,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u8,u16,u32
100,732449,44755,1,1,null,null
100,6568592,1155860,1,0,null,null
100,5411243,1259125,1,1,null,null
100,7371186,1260005,1,0,null,null
100,4943655,1263935,1,0,null,null



val
Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])


uid,item_id,timestamp,event_type,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u8,u16,u32
300,9286415,25907210,1,1,null,null
400,5001732,25900470,1,1,null,null
1100,5905771,25893380,1,0,null,null
1100,9014511,25895020,1,0,null,null
1600,2612724,25896555,1,0,null,null



test
Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])


uid,item_id,timestamp,event_type,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u8,u16,u32
700,3383415,25939545,1,0,null,null
1400,7707460,25993865,1,1,null,null
1400,6005845,25993945,1,1,null,null
1400,7539311,25993945,1,0,null,null
1400,8908728,25993975,1,1,null,null


In [ ]:
for name, lf in [
    ("train", train),
    ("val", val),
    ("test", test),
    ("val_ranking", val_ranking),
    ("test_ranking", test_ranking),
]:
    print(f"\n{name}")
    print(lf.collect_schema())

In [14]:
events_all = pl.concat(
    [events_train, events_val, events_test],
    how="vertical",
)

targets_all = pl.concat(
    [
        train.select("uid", "item_id", "label")
             .with_columns(pl.lit("train").alias("split")),

        val.select("uid", "item_id", "label")
           .with_columns(pl.lit("val").alias("split")),

        test.select("uid", "item_id", "label")
            .with_columns(pl.lit("test").alias("split")),
    ],
    how="vertical",
)

print("Events schema:", events_all.collect_schema())
print("Targets schema:", targets_all.collect_schema())

Events schema: Schema([('uid', UInt32), ('item_id', UInt32), ('timestamp', UInt32), ('event_type', UInt8), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])
Targets schema: Schema([('uid', UInt32), ('item_id', UInt32), ('label', Int8), ('split', String)])


In [15]:
pair_times = (
    events_all
    .group_by(["uid", "item_id"])
    .agg(
        pl.len().alias("n_events"),
        pl.col("timestamp").n_unique().alias("n_timestamps"),
        pl.col("timestamp").min().alias("first_ts"),
        pl.col("timestamp").max().alias("last_ts"),
    )
)

pair_diagnostics = (
    targets_all
    .join(
        pair_times,
        on=["uid", "item_id"],
        how="left",
    )
    .with_columns(
        pl.col("n_events").fill_null(0),
        pl.col("n_timestamps").fill_null(0),
    )
    .group_by(["split", "label"])
    .agg(
        pl.len().alias("n_targets"),

        (pl.col("n_events") == 0)
            .sum().alias("no_event"),

        (pl.col("n_timestamps") == 1)
            .sum().alias("one_timestamp"),

        (pl.col("n_timestamps") > 1)
            .sum().alias("multiple_timestamps"),
    )
    .sort(["split", "label"])
)

display(pair_diagnostics.collect())

split,label,n_targets,no_event,one_timestamp,multiple_timestamps
str,i8,u32,u32,u32,u32
"""test""",0,24617,0,12421,12196
"""test""",1,3885,0,932,2953
"""train""",0,2325287,0,1819693,505594
"""train""",1,836152,0,134255,701897
"""val""",0,22910,0,11059,11851
"""val""",1,4213,0,802,3411


In [16]:
likes_all = pl.concat(
    [likes_train, likes_val, likes_test],
    how="vertical",
)

positive_times = (
    likes_all
    .group_by(["uid", "item_id"])
    .agg(
        pl.len().alias("n_likes"),
        pl.col("timestamp").min().alias("first_like_ts"),
        pl.col("timestamp").max().alias("last_like_ts"),
        pl.col("timestamp").n_unique().alias("n_like_times"),
    )
)

positive_diagnostics = (
    targets_all
    .filter(pl.col("label") == 1)
    .join(
        positive_times,
        on=["uid", "item_id"],
        how="left",
    )
    .group_by("split")
    .agg(
        pl.len().alias("n_positive"),

        pl.col("n_likes").is_not_null()
            .sum().alias("found_in_likes"),

        (pl.col("n_like_times") == 1)
            .sum().alias("unique_like_time"),

        (pl.col("n_like_times") > 1)
            .sum().alias("multiple_like_times"),

        pl.col("n_likes").is_null()
            .sum().alias("missing_in_likes"),
    )
    .sort("split")
)

display(positive_diagnostics.collect())

split,n_positive,found_in_likes,unique_like_time,multiple_like_times,missing_in_likes
str,u32,u32,u32,u32,u32
"""test""",3885,3885,3857,28,0
"""train""",836152,836152,811316,24836,0
"""val""",4213,4213,4161,52,0


In [17]:
def attach_positive_timestamps(targets, likes):
    positive = targets.filter(pl.col("label") == 1)

    first_likes = (
        likes
        .group_by(["uid", "item_id"])
        .agg(
            pl.col("timestamp").min().alias("target_ts"),
            pl.len().alias("n_likes"),
        )
    )

    return (
        positive
        .join(
            first_likes,
            on=["uid", "item_id"],
            how="left",
        )
    )


positive_train = attach_positive_timestamps(
    train, likes_train
)

positive_val = attach_positive_timestamps(
    val, likes_val
)

positive_test = attach_positive_timestamps(
    test, likes_test
)

In [18]:
for name, lf in [
    ("train", positive_train),
    ("val", positive_val),
    ("test", positive_test),
]:
    result = lf.select(
        pl.len().alias("n_positive"),
        pl.col("target_ts").null_count().alias("missing_ts"),
        pl.col("target_ts").min().alias("min_ts"),
        pl.col("target_ts").max().alias("max_ts"),
        (pl.col("n_likes") > 1).sum().alias("repeated_likes"),
    ).collect()

    print(f"\n{name}")
    display(result)


train


n_positive,missing_ts,min_ts,max_ts,repeated_likes
u32,u32,u32,u32,u32
836152,0,80,25823505,27625



val


n_positive,missing_ts,min_ts,max_ts,repeated_likes
u32,u32,u32,u32,u32
4213,0,25825425,25911735,56



test


n_positive,missing_ts,min_ts,max_ts,repeated_likes
u32,u32,u32,u32,u32
3885,0,25913600,25999970,34


### Строим истории

In [19]:
MAX_HISTORY = 50

HISTORY_COLUMNS = [
    "item_id",
    "event_type",
    "timestamp",
    "is_organic",
    "played_ratio_pct",
    "track_length_seconds",
]

history_sources = {
    "train": events_train,
    "val": pl.concat(
        [events_train, events_val],
        how="vertical",
    ),
    "test": pl.concat(
        [events_train, events_val, events_test],
        how="vertical",
    ),
}

positive_sources = {
    "train": positive_train,
    "val": positive_val,
    "test": positive_test,
}

In [20]:
def build_history_sample(
    targets: pl.LazyFrame,
    events: pl.LazyFrame,
    n_sample: int = 100,
    max_history: int = 50,
):
    # Берём небольшую случайную выборку целей.
    sample = (
        targets
        .select("uid", "item_id", "target_ts")
        .limit(n_sample)
        .with_row_index("target_id")
        .collect()
        .lazy()
    )

    users = sample.select("uid").unique()

    relevant_events = events.join(
        users,
        on="uid",
        how="semi",
    )

    # Временной join только для диагностической выборки.
    history = (
        sample
        .join(
            relevant_events,
            on="uid",
            how="inner",
            suffix="_hist",
        )
        .filter(
            pl.col("timestamp") < pl.col("target_ts")
        )
        .sort(
            ["target_id", "timestamp", "item_id_hist"]
        )
        .group_by("target_id", maintain_order=True)
        .tail(max_history)
        .group_by("target_id")
        .agg(
            pl.col("item_id_hist").alias("history_item_ids"),
            pl.col("event_type").alias("history_event_types"),
            pl.col("timestamp").alias("history_timestamps"),
            pl.col("is_organic").alias("history_is_organic"),
            pl.col("played_ratio_pct").alias("history_played_ratio"),
            pl.col("track_length_seconds").alias("history_track_length"),
        )
    )

    result = (
        sample
        .join(
            history,
            on="target_id",
            how="left",
        )
        .sort("target_id")
        .collect()
    )

    return result

ПРоверю на трейне для начала

In [21]:
sample_histories = build_history_sample(
    targets=positive_train,
    events=events_train,
    n_sample=100,
    max_history=MAX_HISTORY,
)

display(
    sample_histories.select(
        "uid",
        "item_id",
        "target_ts",
        "history_item_ids",
        "history_event_types",
    ).head(10)
)

uid,item_id,target_ts,history_item_ids,history_event_types
u32,u32,u32,list[u32],list[u8]
2200,3877385,23702530,"[6762774, 2119435, … 4654254]","[3, 3, … 3]"
2300,8412199,16717945,"[7756989, 1300703, … 8412199]","[3, 3, … 3]"
2700,9196342,14499310,"[9003773, 129160, … 1154896]","[3, 3, … 3]"
2700,2030931,23466680,"[8439805, 7897036, … 2763696]","[3, 3, … 3]"
2800,9378983,19915320,"[4055654, 9058133, … 9378983]","[3, 3, … 3]"
2800,4425201,19947320,"[4875534, 5016149, … 2973068]","[3, 1, … 3]"
2800,3462837,20468700,"[1034521, 1144292, … 5715783]","[3, 3, … 3]"
2800,9051166,20597540,"[6265477, 6642390, … 7295903]","[3, 3, … 3]"
2800,4100952,21237330,"[3968387, 4297258, … 9334263]","[3, 3, … 3]"


#### Есть утечки?

In [22]:
checks = (
    sample_histories
    .with_columns(
        pl.col("history_item_ids")
          .list.len()
          .fill_null(0)
          .alias("history_len"),

        pl.col("history_timestamps")
          .list.max()
          .alias("last_history_ts"),
    )
    .select(
        pl.len().alias("n_samples"),

        pl.col("history_len").max().alias("max_history_len"),

        (pl.col("history_len") == 0)
          .sum().alias("empty_histories"),

        (
            pl.col("last_history_ts").is_not_null()
            & (
                pl.col("last_history_ts")
                >= pl.col("target_ts")
            )
        ).sum().alias("future_leakage"),
    )
)

display(checks)

assert sample_histories.height == 100

assert (
    sample_histories["history_item_ids"]
    .list.len()
    .fill_null(0)
    .max()
    <= MAX_HISTORY
)

assert checks["future_leakage"][0] == 0

print("All checks passed!")

n_samples,max_history_len,empty_histories,future_leakage
u32,u32,u32,u32
100,50,0,0


All checks passed!


In [23]:
from pathlib import Path
import numpy as np
import polars as pl
import gc

MAX_HISTORY = 50
N_BUCKETS = 64

OUTPUT_DIR = Path("./lewis_game_data/histories")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_FEATURES = [
    "item_id",
    "event_type",
    "is_organic",
    "played_ratio_pct",
    "track_length_seconds",
]

In [24]:
def build_history_bucket(
    targets: pl.LazyFrame,
    events: pl.LazyFrame,
    bucket: int,
    n_buckets: int = 64,
    max_history: int = 50,
):
    # Цели конкретной группы пользователей
    t = (
        targets
        .filter(pl.col("uid") % n_buckets == bucket)
        .select("uid", "item_id", "target_ts")
        .collect()
    )

    if t.is_empty():
        return None

    # Исторические события той же группы
    e = (
        events
        .filter(pl.col("uid") % n_buckets == bucket)
        .select(
            "uid",
            "item_id",
            "timestamp",
            "event_type",
            "is_organic",
            "played_ratio_pct",
            "track_length_seconds",
        )
        .collect()
    )

    # Сортируем события по (uid, timestamp).
    e = e.sort(["uid", "timestamp", "item_id", "event_type"])

    uid_e = e["uid"].to_numpy().astype(np.uint64)
    ts_e = e["timestamp"].to_numpy().astype(np.uint64)

    uid_t = t["uid"].to_numpy().astype(np.uint64)
    ts_t = t["target_ts"].to_numpy().astype(np.uint64)

    # Составной ключ сохраняет лексикографический порядок.
    # timestamp и uid имеют тип UInt32.
    event_keys = (uid_e << np.uint64(32)) | ts_e
    target_keys = (uid_t << np.uint64(32)) | ts_t

    # side="left" исключает события с timestamp >= target_ts.
    ends = np.searchsorted(
        event_keys,
        target_keys,
        side="left",
    )

    # Начало истории данного пользователя.
    user_starts = np.searchsorted(
        event_keys,
        uid_t << np.uint64(32),
        side="left",
    )

    # Индексы последних max_history событий.
    offsets = np.arange(max_history, dtype=np.int64)

    indices = (
        ends[:, None]
        - max_history
        + offsets[None, :]
    )

    valid = (
        (indices >= user_starts[:, None])
        & (indices < ends[:, None])
    )

    # На случай пустых историй и начала массива.
    safe_indices = np.clip(indices, 0, max(len(e) - 1, 0))

    output = t.select("uid", "item_id", "target_ts")

    feature_specs = {
        "item_id": ("history_item_ids", 1),
        "event_type": ("history_event_types", 0),
        "is_organic": ("history_is_organic", 0),
        "played_ratio_pct": ("history_played_ratio", 0),
        "track_length_seconds": ("history_track_length", 0),
    }

    for source_col, (out_col, shift) in feature_specs.items():
        if len(e) == 0:
            values = np.zeros(indices.shape, dtype=np.int64)
        else:
            arr = (
                e[source_col]
                .fill_null(0)
                .to_numpy()
                .astype(np.int64)
            )

            values = np.where(
                valid,
                arr[safe_indices] + shift,
                0,
            )

        output = output.with_columns(
            pl.Series(
                out_col,
                values.tolist(),
            )
        )

    # Расстояние по времени от целевого события.
    if len(e) == 0:
        deltas = np.zeros(indices.shape, dtype=np.int64)
    else:
        deltas = np.where(
            valid,
            ts_t[:, None].astype(np.int64)
            - ts_e[safe_indices].astype(np.int64),
            0,
        )

    output = output.with_columns(
        pl.Series(
            "history_time_deltas",
            deltas.tolist(),
        ),
        pl.Series(
            "history_len",
            valid.sum(axis=1).astype(np.uint16),
        ),
    )

    return output

In [25]:
bucket_0 = build_history_bucket(
    targets=positive_train,
    events=events_train,
    bucket=0,
    n_buckets=N_BUCKETS,
    max_history=MAX_HISTORY,
)

print("Rows:", bucket_0.height)
display(bucket_0.head(5))

Rows: 54829


uid,item_id,target_ts,history_item_ids,history_event_types,history_is_organic,history_played_ratio,history_track_length,history_time_deltas,history_len
u32,u32,u32,list[i64],list[i64],list[i64],list[i64],list[i64],list[i64],u16
3200,6901374,12057555,"[1497918, 4457960, … 5744742]","[3, 3, … 3]","[1, 1, … 0]","[100, 100, … 3]","[180, 190, … 150]","[12730, 12540, … 30]",50
3200,826189,12057555,"[1497918, 4457960, … 5744742]","[3, 3, … 3]","[1, 1, … 0]","[100, 100, … 3]","[180, 190, … 150]","[12730, 12540, … 30]",50
3200,5635052,12057555,"[1497918, 4457960, … 5744742]","[3, 3, … 3]","[1, 1, … 0]","[100, 100, … 3]","[180, 190, … 150]","[12730, 12540, … 30]",50
3200,6155741,18663030,"[4334790, 7810757, … 1669556]","[3, 3, … 3]","[1, 1, … 0]","[100, 100, … 1]","[240, 170, … 255]","[15345, 15345, … 10]",50
6400,1502556,935280,"[0, 0, … 5969466]","[0, 0, … 1]","[0, 0, … 1]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 6770]",2


In [26]:
check = bucket_0.select(
    pl.col("history_len").max().alias("max_len"),
    pl.col("history_len").min().alias("min_len"),

    (
        pl.col("history_time_deltas")
        .list.eval(pl.element() < 0)
        .list.sum()
        .sum()
    ).alias("negative_deltas"),

    (
        pl.col("history_item_ids")
        .list.eval(pl.element() != 0)
        .list.sum()
        != pl.col("history_len")
    ).sum().alias("length_mismatch"),
)

display(check)

assert check["max_len"][0] <= MAX_HISTORY
assert check["negative_deltas"][0] == 0
assert check["length_mismatch"][0] == 0

print("Bucket checks passed!")

max_len,min_len,negative_deltas,length_mismatch
u16,u16,u32,u32
50,0,0,0


Bucket checks passed!


In [27]:
assert (
    bucket_0
    .select(
        pl.struct([
            "history_item_ids",
            "history_time_deltas",
        ])
        .map_elements(
            lambda x: all(
                delta > 0
                for item, delta in zip(
                    x["history_item_ids"],
                    x["history_time_deltas"],
                )
                if item != 0
            ),
            return_dtype=pl.Boolean,
        )
        .all()
    )
    .item()
)

print("Strict temporal check passed!")

Strict temporal check passed!


In [28]:
def save_history_buckets(
    targets,
    events,
    split_name,
    n_buckets=N_BUCKETS,
    max_history=MAX_HISTORY,
):
    split_dir = OUTPUT_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    total_rows = 0

    for bucket in range(n_buckets):
        out_path = split_dir / f"part_{bucket:03d}.parquet"

        if out_path.exists():
            df = pl.read_parquet(
                out_path,
                columns=["uid"],
            )
            total_rows += len(df)
            continue

        df = build_history_bucket(
            targets=targets,
            events=events,
            bucket=bucket,
            n_buckets=n_buckets,
            max_history=max_history,
        )

        if df is None:
            continue

        df.write_parquet(
            out_path,
            compression="zstd",
        )

        total_rows += df.height

        if bucket % 8 == 0:
            print(
                f"{split_name}: "
                f"{bucket + 1}/{n_buckets}, "
                f"saved rows = {total_rows:,}"
            )

        del df
        gc.collect()

    print(f"{split_name} completed: {total_rows:,} rows")

    return pl.scan_parquet(
        str(split_dir / "*.parquet")
    )

In [29]:
game_histories_train = save_history_buckets(
    targets=positive_train,
    events=history_sources["train"],
    split_name="train",
)

train: 1/64, saved rows = 54,829
train: 9/64, saved rows = 161,451
train: 17/64, saved rows = 273,454
train: 25/64, saved rows = 373,377
train: 33/64, saved rows = 474,012
train: 41/64, saved rows = 571,683
train: 49/64, saved rows = 677,710
train: 57/64, saved rows = 792,577
train completed: 836,152 rows


In [30]:
n_expected = positive_train.select(pl.len()).collect().item()
n_actual = game_histories_train.select(pl.len()).collect().item()

print("Expected:", n_expected)
print("Actual:", n_actual)

assert n_expected == n_actual

Expected: 836152
Actual: 836152


In [31]:
game_histories_val = save_history_buckets(
    targets=positive_val,
    events=history_sources["val"],
    split_name="val",
)

game_histories_test = save_history_buckets(
    targets=positive_test,
    events=history_sources["test"],
    split_name="test",
)

val: 1/64, saved rows = 224
val: 9/64, saved rows = 848
val: 17/64, saved rows = 1,458
val: 25/64, saved rows = 1,951
val: 33/64, saved rows = 2,385
val: 41/64, saved rows = 2,905
val: 49/64, saved rows = 3,430
val: 57/64, saved rows = 3,939
val completed: 4,213 rows
test: 1/64, saved rows = 299
test: 9/64, saved rows = 782
test: 17/64, saved rows = 1,330
test: 25/64, saved rows = 1,842
test: 33/64, saved rows = 2,266
test: 41/64, saved rows = 2,711
test: 49/64, saved rows = 3,134
test: 57/64, saved rows = 3,641
test completed: 3,885 rows


Распределение длин историй

In [32]:
def history_length_stats(lf):
    return lf.select(
        pl.len().alias("n_examples"),
        pl.col("history_len").mean().alias("mean_len"),
        pl.col("history_len").median().alias("median_len"),
        pl.col("history_len").quantile(0.10).alias("p10"),
        pl.col("history_len").quantile(0.90).alias("p90"),
        (pl.col("history_len") == 0).sum().alias("empty"),
        (pl.col("history_len") == 50).sum().alias("full_50"),
    ).collect()


for name, lf in [
    ("train", game_histories_train),
    ("val", game_histories_val),
    ("test", game_histories_test),
]:
    print(f"\n{name}")
    display(history_length_stats(lf))


train


n_examples,mean_len,median_len,p10,p90,empty,full_50
u32,f64,f64,f64,f64,u32,u32
836152,48.115838,50.0,50.0,50.0,6256,784053



val


n_examples,mean_len,median_len,p10,p90,empty,full_50
u32,f64,f64,f64,f64,u32,u32
4213,49.737242,50.0,50.0,50.0,0,4155



test


n_examples,mean_len,median_len,p10,p90,empty,full_50
u32,f64,f64,f64,f64,u32,u32
3885,49.685714,50.0,50.0,50.0,0,3824


In [33]:
item_catalog = (
    events_train
    .group_by("item_id")
    .agg(
        pl.col("timestamp").min().alias("first_seen_ts")
    )
    .sort("first_seen_ts", "item_id")
    .collect()
)

print("Number of items:", item_catalog.height)

display(item_catalog.head(10))

Number of items: 882683


item_id,first_seen_ts
u32,u32
7400764,0
3415205,5
6728180,10
5283544,15
8932363,15
140621,20
1652612,20
5638983,20
6099388,20


In [34]:
catalog_times = item_catalog["first_seen_ts"].to_numpy()

def catalog_availability(lf, name):
    targets = lf.select(
        "target_ts"
    ).collect()

    ts = targets["target_ts"].to_numpy()

    n_available = np.searchsorted(
        catalog_times,
        ts,
        side="left",
    )

    print(f"\n{name}")
    print("Min available:", n_available.min())
    print("Median available:", np.median(n_available))
    print("Max available:", n_available.max())
    print("Fewer than 4:", (n_available < 4).sum())


catalog_availability(game_histories_train, "train")
catalog_availability(game_histories_val, "val")
catalog_availability(game_histories_test, "test")


train
Min available: 39
Median available: 614784.0
Max available: 882679
Fewer than 4: 0

val
Min available: 882683
Median available: 882683.0
Max available: 882683
Fewer than 4: 0

test
Min available: 882683
Median available: 882683.0
Max available: 882683
Fewer than 4: 0


Выбираем кандидатов

In [35]:
{
    "candidate_item_ids": [49124, 8162, 732449, 98123],
    "target_position": 2,
}

{'candidate_item_ids': [49124, 8162, 732449, 98123], 'target_position': 2}

In [36]:
N_CANDIDATES = 4
N_NEGATIVES = N_CANDIDATES - 1
SEED = 42

# Каталог уже отсортирован по first_seen_ts
catalog_ids = item_catalog["item_id"].to_numpy()
catalog_times = item_catalog["first_seen_ts"].to_numpy()

assert np.all(catalog_times[:-1] <= catalog_times[1:])
assert len(np.unique(catalog_ids)) == len(catalog_ids)

# Используем для очистки кандидатов от известных позитивов.
# Не передаём эту информацию нейросетям.
known_likes = pl.concat(
    [
        likes_train.select("uid", "item_id"),
        likes_val.select("uid", "item_id"),
        likes_test.select("uid", "item_id"),
    ],
    how="vertical",
).unique()

def pair_key(uid, item_id):
    """
    Упаковывает пару UInt32, UInt32 в один UInt64.
    """
    return (
        (np.asarray(uid, dtype=np.uint64) << np.uint64(32))
        | np.asarray(item_id, dtype=np.uint64)
    )

Сначала попробуем

In [37]:
def sample_candidates_batch(
    df: pl.DataFrame,
    forbidden_keys: np.ndarray,
    rng: np.random.Generator,
    n_candidates: int = 4,
    max_rounds: int = 100,
):
    """
    Создаёт candidate_item_ids и target_position.

    forbidden_keys — отсортированный массив UInt64
    с известными положительными парами.

    history_item_ids уже содержат item_id + 1,
    где 0 используется для PAD.
    """
    n = df.height
    n_neg = n_candidates - 1

    if n == 0:
        return df

    uid = df["uid"].to_numpy().astype(np.uint64)
    positive = df["item_id"].to_numpy().astype(np.uint32)
    ts = df["target_ts"].to_numpy()

    # Доступный каталог для каждой цели
    available = np.searchsorted(
        catalog_times, ts, side="left"
    )

    if np.any(available < n_candidates):
        raise ValueError("Not enough available items")

    history = np.asarray(
        df["history_item_ids"].to_list(),
        dtype=np.uint64,
    )

    negatives = np.zeros(
        (n, n_neg), dtype=np.uint32
    )

    filled = np.zeros(
        (n, n_neg), dtype=bool
    )

    # Отдельно формируем каждый отрицательный кандидат
    for j in range(n_neg):
        pending = np.arange(n)

        for _ in range(max_rounds):
            if len(pending) == 0:
                break

            # Uniform sampling по доступному каталогу
            indices = (
                rng.random(len(pending))
                * available[pending]
            ).astype(np.int64)

            sampled = catalog_ids[indices].astype(np.uint32)

            # Исключаем текущий позитив
            valid = sampled != positive[pending]

            # Исключаем любой трек из входной истории
            valid &= ~np.any(
                history[pending] == (
                    sampled[:, None].astype(np.uint64) + 1
                ),
                axis=1,
            )

            # Исключаем ранее выбранные негативы
            if j > 0:
                valid &= ~np.any(
                    negatives[pending, :j] == sampled[:, None],
                    axis=1,
                )

            # Проверка по известным лайкам
            keys = pair_key(uid[pending], sampled)
            positions = np.searchsorted(forbidden_keys, keys)

            safe_positions = np.minimum(
                positions,
                len(forbidden_keys) - 1,
            )

            is_known_like = (
                forbidden_keys[safe_positions] == keys
            )

            valid &= ~is_known_like

            accepted = pending[valid]

            negatives[accepted, j] = sampled[valid]
            filled[accepted, j] = True

            pending = pending[~valid]

        if len(pending) > 0:
            raise RuntimeError(
                f"Could not sample negatives for "
                f"{len(pending)} examples"
            )

    # Добавляем положительный трек
    candidates = np.column_stack(
        [positive, negatives]
    )

    # Перемешиваем кандидатов независимо в каждой строке
    permutations = np.argsort(
        rng.random((n, n_candidates)),
        axis=1,
    )

    shuffled = np.take_along_axis(
        candidates, permutations, axis=1
    )

    target_position = np.argmax(
        permutations == 0,
        axis=1,
    ).astype(np.uint8)

    return df.with_columns(
        pl.Series(
            "candidate_item_ids",
            shuffled.tolist(),
        ),
        pl.Series(
            "target_position",
            target_position,
        ),
    )

In [38]:
bucket = 0

# Истории уже сохранены по bucket
sample = (
    pl.read_parquet(
        OUTPUT_DIR / "train" / f"part_{bucket:03d}.parquet"
    )
    .head(1000)
)

# Извлекаем только лайки пользователей текущего bucket
forbidden = (
    known_likes
    .filter(pl.col("uid") % N_BUCKETS == bucket)
    .collect()
)

forbidden_keys = np.unique(
    pair_key(
        forbidden["uid"].to_numpy(),
        forbidden["item_id"].to_numpy(),
    )
)

rng = np.random.default_rng(SEED)

candidate_sample = sample_candidates_batch(
    sample,
    forbidden_keys,
    rng,
)

display(
    candidate_sample.select(
        "uid",
        "item_id",
        "candidate_item_ids",
        "target_position",
    ).head(10)
)

uid,item_id,candidate_item_ids,target_position
u32,u32,list[i64],u8
3200,6901374,"[6753046, 6901374, … 404518]",1
3200,826189,"[4066113, 6154715, … 1181404]",2
3200,5635052,"[5635052, 6550559, … 7901988]",0
3200,6155741,"[7643645, 6155741, … 2335369]",1
6400,1502556,"[9241602, 1502556, … 7208385]",1
6400,4926054,"[8569130, 1443972, … 5129697]",2
6400,6281017,"[6281017, 6587395, … 2697591]",0
6400,8426027,"[8604813, 3370969, … 8426027]",3
6400,480120,"[8793425, 2996426, … 480120]",3


In [39]:
positive = candidate_sample["item_id"].to_numpy()

candidates = np.asarray(
    candidate_sample["candidate_item_ids"].to_list(),
    dtype=np.uint32,
)

positions = candidate_sample["target_position"].to_numpy()

# Ровно четыре кандидата
assert candidates.shape == (len(positive), 4)

# Все кандидаты различны
assert all(
    len(np.unique(row)) == 4
    for row in candidates
)

# Правильный ответ находится в target_position
assert np.all(
    candidates[np.arange(len(positive)), positions]
    == positive
)

# Негативы не были известными лайками
uid = candidate_sample["uid"].to_numpy()
negative_mask = np.ones_like(candidates, dtype=bool)
negative_mask[np.arange(len(positive)), positions] = False

negative_keys = pair_key(
    np.broadcast_to(uid[:, None], candidates.shape)[negative_mask],
    candidates[negative_mask],
)

assert not np.isin(
    negative_keys, forbidden_keys
).any()

# Кандидаты должны были появиться до target_ts
first_seen = dict(
    zip(
        item_catalog["item_id"].to_list(),
        item_catalog["first_seen_ts"].to_list(),
    )
)

for row, ts, pos in zip(
    candidates,
    candidate_sample["target_ts"].to_numpy(),
    positions,
):
    for j, item in enumerate(row):
        if j != pos:
            assert first_seen[int(item)] < ts

print("Candidate checks passed!")

Candidate checks passed!


### Создаем датасет для игры

In [41]:
import gc
import numpy as np
import polars as pl

from pathlib import Path

GAME_DIR = Path("./lewis_game_data/game_a1")
GAME_DIR.mkdir(parents=True, exist_ok=True)

N_BUCKETS = 64
N_CANDIDATES = 4
SEED = 42

SPLIT_CODES = {
    "train": 0,
    "val": 1,
    "test": 2,
}

EXPECTED_COUNTS = {
    "train": 836_152,
    "val": 4_213,
    "test": 3_885,
}

In [43]:
def build_game_dataset(
    split_name: str,
    n_buckets: int = N_BUCKETS,
    overwrite: bool = False,
):
    """
    Строит полный датасет referential game.

    Использует:
      - OUTPUT_DIR: истории из предыдущего этапа
      - GAME_DIR: выходные Parquet-файлы
      - known_likes: пары известных лайков
      - sample_candidates_batch()
    """

    assert split_name in SPLIT_CODES

    input_dir = OUTPUT_DIR / split_name
    output_dir = GAME_DIR / split_name

    output_dir.mkdir(parents=True, exist_ok=True)

    total_rows = 0

    for bucket in range(n_buckets):

        input_path = (
            input_dir / f"part_{bucket:03d}.parquet"
        )

        output_path = (
            output_dir / f"part_{bucket:03d}.parquet"
        )

        if not input_path.exists():
            if output_path.exists() and not overwrite:
                total_rows += (
                    pl.scan_parquet(output_path)
                    .select(pl.len())
                    .collect()
                    .item()
                )
                continue

            # Пустой bucket
            continue

        if output_path.exists() and not overwrite:
            n_rows = (
                pl.scan_parquet(output_path)
                .select(pl.len())
                .collect()
                .item()
            )

            total_rows += n_rows
            continue

        # 1. Загружаем истории текущего bucket
        df = pl.read_parquet(input_path)

        if df.is_empty():
            continue

        # 2. Лайки пользователей текущего bucket
        forbidden_df = (
            known_likes
            .filter(
                pl.col("uid") % n_buckets == bucket
            )
            .collect()
        )

        forbidden_keys = np.unique(
            pair_key(
                forbidden_df["uid"].to_numpy(),
                forbidden_df["item_id"].to_numpy(),
            )
        )

        # 3. Независимый генератор для каждого bucket
        rng = np.random.default_rng(
            SEED
            + SPLIT_CODES[split_name] * 100_000
            + bucket
        )

        # 4. Генерируем кандидатов
        result = sample_candidates_batch(
            df=df,
            forbidden_keys=forbidden_keys,
            rng=rng,
            n_candidates=N_CANDIDATES,
        )

        # 5. Проверяем количество строк
        assert result.height == df.height

        # 6. Сохраняем
        # Сначала временный файл, затем переименовываем.
        temp_path = output_path.with_suffix(
            ".parquet.tmp"
        )

        result.write_parquet(
            temp_path,
            compression="zstd",
        )

        temp_path.replace(output_path)

        total_rows += result.height

        if bucket % 8 == 0:
            print(
                f"{split_name}: "
                f"{bucket + 1}/{n_buckets} buckets, "
                f"{total_rows:,} examples"
            )

        del df, result, forbidden_df, forbidden_keys
        gc.collect()

    print(
        f"{split_name} completed: "
        f"{total_rows:,} examples"
    )

    assert total_rows == EXPECTED_COUNTS[split_name], (
        f"{split_name}: expected "
        f"{EXPECTED_COUNTS[split_name]}, "
        f"got {total_rows}"
    )

    return pl.scan_parquet(
        str(output_dir / "part_*.parquet")
    )

In [44]:
def build_game_dataset(
    split_name: str,
    n_buckets: int = N_BUCKETS,
    overwrite: bool = False,
):
    """
    Строит полный датасет referential game.

    Использует:
      - OUTPUT_DIR: истории из предыдущего этапа
      - GAME_DIR: выходные Parquet-файлы
      - known_likes: пары известных лайков
      - sample_candidates_batch()
    """

    assert split_name in SPLIT_CODES

    input_dir = OUTPUT_DIR / split_name
    output_dir = GAME_DIR / split_name

    output_dir.mkdir(parents=True, exist_ok=True)

    total_rows = 0

    for bucket in range(n_buckets):

        input_path = (
            input_dir / f"part_{bucket:03d}.parquet"
        )

        output_path = (
            output_dir / f"part_{bucket:03d}.parquet"
        )

        if not input_path.exists():
            if output_path.exists() and not overwrite:
                total_rows += (
                    pl.scan_parquet(output_path)
                    .select(pl.len())
                    .collect()
                    .item()
                )
                continue

            # Пустой bucket
            continue

        if output_path.exists() and not overwrite:
            n_rows = (
                pl.scan_parquet(output_path)
                .select(pl.len())
                .collect()
                .item()
            )

            total_rows += n_rows
            continue

        # 1. Загружаем истории текущего bucket
        df = pl.read_parquet(input_path)

        if df.is_empty():
            continue

        # 2. Лайки пользователей текущего bucket
        forbidden_df = (
            known_likes
            .filter(
                pl.col("uid") % n_buckets == bucket
            )
            .collect()
        )

        forbidden_keys = np.unique(
            pair_key(
                forbidden_df["uid"].to_numpy(),
                forbidden_df["item_id"].to_numpy(),
            )
        )

        # 3. Независимый генератор для каждого bucket
        rng = np.random.default_rng(
            SEED
            + SPLIT_CODES[split_name] * 100_000
            + bucket
        )

        # 4. Генерируем кандидатов
        result = sample_candidates_batch(
            df=df,
            forbidden_keys=forbidden_keys,
            rng=rng,
            n_candidates=N_CANDIDATES,
        )

        # 5. Проверяем количество строк
        assert result.height == df.height

        # 6. Сохраняем
        # Сначала временный файл, затем переименовываем.
        temp_path = output_path.with_suffix(
            ".parquet.tmp"
        )

        result.write_parquet(
            temp_path,
            compression="zstd",
        )

        temp_path.replace(output_path)

        total_rows += result.height

        if bucket % 8 == 0:
            print(
                f"{split_name}: "
                f"{bucket + 1}/{n_buckets} buckets, "
                f"{total_rows:,} examples"
            )

        del df, result, forbidden_df, forbidden_keys
        gc.collect()

    print(
        f"{split_name} completed: "
        f"{total_rows:,} examples"
    )

    assert total_rows == EXPECTED_COUNTS[split_name], (
        f"{split_name}: expected "
        f"{EXPECTED_COUNTS[split_name]}, "
        f"got {total_rows}"
    )

    return pl.scan_parquet(
        str(output_dir / "part_*.parquet")
    )

In [45]:
game_train = build_game_dataset(
    split_name="train",
)

train: 1/64 buckets, 54,829 examples
train: 9/64 buckets, 161,451 examples
train: 17/64 buckets, 273,454 examples
train: 25/64 buckets, 373,377 examples
train: 33/64 buckets, 474,012 examples
train: 41/64 buckets, 571,683 examples
train: 49/64 buckets, 677,710 examples
train: 57/64 buckets, 792,577 examples
train completed: 836,152 examples


In [46]:
game_val = build_game_dataset(
    split_name="val",
)

game_test = build_game_dataset(
    split_name="test",
)

val: 1/64 buckets, 224 examples
val: 9/64 buckets, 848 examples
val: 17/64 buckets, 1,458 examples
val: 25/64 buckets, 1,951 examples
val: 33/64 buckets, 2,385 examples
val: 41/64 buckets, 2,905 examples
val: 49/64 buckets, 3,430 examples
val: 57/64 buckets, 3,939 examples
val completed: 4,213 examples
test: 1/64 buckets, 299 examples
test: 9/64 buckets, 782 examples
test: 17/64 buckets, 1,330 examples
test: 25/64 buckets, 1,842 examples
test: 33/64 buckets, 2,266 examples
test: 41/64 buckets, 2,711 examples
test: 49/64 buckets, 3,134 examples
test: 57/64 buckets, 3,641 examples
test completed: 3,885 examples


In [ ]:
def game_statistics(lf, name):
    result = (
        lf.select(
            pl.len().alias("n_examples"),

            pl.col("history_len")
              .mean().alias("mean_history_len"),

            pl.col("history_len")
              .max().alias("max_history_len"),

            (pl.col("history_len") == 0)
              .sum().alias("empty_histories"),

            pl.col("candidate_item_ids")
              .list.len()
              .min().alias("min_candidates"),

            pl.col("candidate_item_ids")
              .list.len()
              .max().alias("max_candidates"),
        )
        .collect()
    )

    print(f"\n{name}")
    display(result)


for name, lf in [
    ("train", game_train),
    ("val", game_val),
    ("test", game_test),
]:
    game_statistics(lf, name)

Позиции правильны ответов

In [47]:
for name, lf in [
    ("train", game_train),
    ("val", game_val),
    ("test", game_test),
]:

    distribution = (
        lf.group_by("target_position")
        .agg(pl.len().alias("n"))
        .with_columns(
            (
                pl.col("n") / pl.col("n").sum()
            ).alias("fraction")
        )
        .sort("target_position")
        .collect()
    )

    print(f"\n{name}")
    display(distribution)


train


target_position,n,fraction
u8,u32,f64
0,209515,0.25057
1,208778,0.249689
2,209290,0.250301
3,208569,0.249439



val


target_position,n,fraction
u8,u32,f64
0,1075,0.255163
1,1072,0.254451
2,1043,0.247567
3,1023,0.24282



test


target_position,n,fraction
u8,u32,f64
0,992,0.255341
1,962,0.247619
2,963,0.247876
3,968,0.249163


Позиция трека

In [48]:
for name, lf in [
    ("train", game_train),
    ("val", game_val),
    ("test", game_test),
]:
    checks = (
        lf.select(
            pl.len().alias("total"),

            (
                pl.col("candidate_item_ids")
                .list.get(
                    pl.col("target_position")
                    .cast(pl.UInt32)
                )
                == pl.col("item_id")
            ).sum().alias("correct_positive"),

            (
                pl.col("candidate_item_ids")
                .list.n_unique() == N_CANDIDATES
            ).sum().alias("unique_candidates"),
        )
        .collect()
    )

    print(f"\n{name}")
    display(checks)

    assert checks["correct_positive"][0] == checks["total"][0]
    assert checks["unique_candidates"][0] == checks["total"][0]

print("All game dataset checks passed!")


train


total,correct_positive,unique_candidates
u32,u32,u32
836152,836152,836152



val


total,correct_positive,unique_candidates
u32,u32,u32
4213,4213,4213



test


total,correct_positive,unique_candidates
u32,u32,u32
3885,3885,3885


All game dataset checks passed!


In [49]:
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PAD_ID = 0
UNK_ID = 1

# item_catalog уже существует из предыдущих шагов.
# Только известные на train треки.
vocab_items = np.sort(
    item_catalog["item_id"]
    .to_numpy()
    .astype(np.uint32)
)

assert len(vocab_items) > 0
assert len(np.unique(vocab_items)) == len(vocab_items)

N_ITEMS = len(vocab_items) + 2

print("Number of train items:", len(vocab_items))
print("Embedding vocabulary size:", N_ITEMS)
print("Device:", DEVICE)


def encode_items(raw_ids, padded=False):
    """
    Преобразует исходные item_id в индексы embeddings.

    Если padded=True:
        0 означает PAD,
        все ненулевые значения = original item_id + 1
        (так сохранены history_item_ids).

    Неизвестные item_id -> UNK_ID.
    """
    raw_ids = np.asarray(raw_ids, dtype=np.int64)

    if padded:
        mask_pad = raw_ids == 0
        actual_ids = raw_ids - 1
    else:
        mask_pad = np.zeros(
            raw_ids.shape, dtype=bool
        )
        actual_ids = raw_ids

    positions = np.searchsorted(
        vocab_items,
        actual_ids
    )

    safe = np.clip(
        positions, 0, len(vocab_items) - 1
    )

    found = (
        vocab_items[safe] == actual_ids
    )

    encoded = np.where(
        found,
        safe + 2,
        UNK_ID,
    ).astype(np.int64)

    encoded[mask_pad] = PAD_ID

    return encoded

Number of train items: 882683
Embedding vocabulary size: 882685
Device: cuda


In [50]:
example_ids = vocab_items[:5]

encoded = encode_items(example_ids)

assert np.all(encoded >= 2)
assert len(np.unique(encoded)) == len(encoded)

test_history = np.array(
    [[0, 0, int(example_ids[0]) + 1]]
)

encoded_history = encode_items(
    test_history,
    padded=True
)

assert encoded_history[0, 0] == PAD_ID
assert encoded_history[0, 1] == PAD_ID
assert encoded_history[0, 2] == encoded[0]

print("Item vocabulary checks passed!")

Item vocabulary checks passed!


In [51]:
class LewisGameDataset(Dataset):

    def __init__(self, df: pl.DataFrame):
        self.n = df.height

        self.history_items = torch.from_numpy(
            encode_items(
                np.asarray(
                    df["history_item_ids"].to_list(),
                    dtype=np.int64,
                ),
                padded=True,
            )
        )

        self.history_events = torch.tensor(
            df["history_event_types"].to_list(),
            dtype=torch.long,
        )

        # Интервалы времени преобразуем в log1p.
        # Сырой временной интервал не является
        # вручную вычисленным пользовательским агрегатом.
        deltas = np.asarray(
            df["history_time_deltas"].to_list(),
            dtype=np.float32,
        )

        self.history_deltas = torch.from_numpy(
            np.log1p(deltas)
        )

        self.candidates = torch.from_numpy(
            encode_items(
                np.asarray(
                    df["candidate_item_ids"].to_list(),
                    dtype=np.int64,
                ),
                padded=False,
            )
        )

        self.targets = torch.tensor(
            df["target_position"].to_list(),
            dtype=torch.long,
        )

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return {
            "history_items": self.history_items[idx],
            "history_events": self.history_events[idx],
            "history_deltas": self.history_deltas[idx],
            "candidates": self.candidates[idx],
            "target": self.targets[idx],
        }

In [52]:
from pathlib import Path

smoke_df = (
    pl.read_parquet(
        OUTPUT_DIR / "train" / "part_000.parquet"
    )
    .head(4096)
)

# Важное уточнение:
# здесь нужны уже готовые кандидаты,
# поэтому читаем из GAME_DIR.
smoke_df = (
    pl.read_parquet(
        GAME_DIR / "train" / "part_000.parquet"
    )
    .head(4096)
)

smoke_dataset = LewisGameDataset(smoke_df)

smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

batch = next(iter(smoke_loader))

for name, tensor in batch.items():
    print(name, tuple(tensor.shape), tensor.dtype)

assert batch["history_items"].shape[1] == 50
assert batch["candidates"].shape[1] == 4
assert batch["target"].min() >= 0
assert batch["target"].max() < 4

print("Dataset checks passed!")

history_items (256, 50) torch.int64
history_events (256, 50) torch.int64
history_deltas (256, 50) torch.float32
candidates (256, 4) torch.int64
target (256,) torch.int64
Dataset checks passed!


In [53]:
class Sender(nn.Module):

    def __init__(
        self,
        n_items,
        item_dim=64,
        event_dim=16,
        hidden_dim=128,
        vocab_size=32,
        message_length=4,
        max_history=50,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.message_length = message_length

        self.item_embedding = nn.Embedding(
            n_items,
            item_dim,
            padding_idx=PAD_ID,
        )

        self.event_embedding = nn.Embedding(
            4,
            event_dim,
            padding_idx=0,
        )

        self.position_embedding = nn.Embedding(
            max_history,
            16,
        )

        input_dim = item_dim + event_dim + 16 + 1

        self.event_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.attention = nn.Linear(
            hidden_dim, 1
        )

        self.message_head = nn.Linear(
            hidden_dim,
            message_length * vocab_size,
        )

    def forward(
        self,
        history_items,
        history_events,
        history_deltas,
        temperature=1.0,
    ):
        B, T = history_items.shape

        item_emb = self.item_embedding(
            history_items
        )

        event_emb = self.event_embedding(
            history_events
        )

        positions = torch.arange(
            T,
            device=history_items.device,
        )

        pos_emb = self.position_embedding(
            positions
        ).unsqueeze(0).expand(B, -1, -1)

        x = torch.cat(
            [
                item_emb,
                event_emb,
                pos_emb,
                history_deltas.unsqueeze(-1),
            ],
            dim=-1,
        )

        h = self.event_encoder(x)

        mask = history_items != PAD_ID

        attention_logits = (
            self.attention(h).squeeze(-1)
        )

        attention_logits = (
            attention_logits.masked_fill(
                ~mask, -1e9
            )
        )

        # Для пустой истории избегаем softmax
        # по полностью замаскированным позициям.
        empty = ~mask.any(dim=1)

        attention_logits[empty] = 0.0

        weights = F.softmax(
            attention_logits, dim=1
        )

        weights = weights * mask.float()

        weights = weights / (
            weights.sum(dim=1, keepdim=True)
            .clamp_min(1e-8)
        )

        user_embedding = (
            h * weights.unsqueeze(-1)
        ).sum(dim=1)

        logits = self.message_head(
            user_embedding
        ).reshape(
            B,
            self.message_length,
            self.vocab_size,
        )

        if self.training:
            message = F.gumbel_softmax(
                logits,
                tau=temperature,
                hard=True,
                dim=-1,
            )
        else:
            indices = logits.argmax(dim=-1)

            message = F.one_hot(
                indices,
                num_classes=self.vocab_size,
            ).float()

        return {
            "message": message,
            "message_ids": message.argmax(dim=-1),
            "user_embedding": user_embedding,
            "message_logits": logits,
        }

In [54]:
class Receiver(nn.Module):

    def __init__(
        self,
        n_items,
        item_dim=64,
        hidden_dim=128,
        vocab_size=32,
        message_length=4,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.message_length = message_length

        # Кодовая книга: отдельные значения
        # для каждой позиции сообщения.
        self.codebook = nn.Parameter(
            torch.randn(
                message_length,
                vocab_size,
                hidden_dim,
            ) * 0.02
        )

        self.message_encoder = nn.Sequential(
            nn.Linear(
                message_length * hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(hidden_dim, item_dim),
        )

        self.item_embedding = nn.Embedding(
            n_items,
            item_dim,
            padding_idx=PAD_ID,
        )

        self.item_bias = nn.Embedding(
            n_items, 1
        )

    def forward(self, message, candidates):
        # message: [B, L, V]
        # codebook: [L, V, H]

        symbol_embeddings = torch.einsum(
            "blv,lvh->blh",
            message,
            self.codebook,
        )

        B = message.shape[0]

        z = self.message_encoder(
            symbol_embeddings.reshape(B, -1)
        )

        # [B, K, D]
        candidate_embeddings = (
            self.item_embedding(candidates)
        )

        # Совместимость user message и item.
        scores = torch.einsum(
            "bd,bkd->bk",
            z,
            candidate_embeddings,
        ) / (z.shape[-1] ** 0.5)

        scores = scores + (
            self.item_bias(candidates)
            .squeeze(-1)
        )

        return scores

In [55]:
class LewisGame(nn.Module):

    def __init__(
        self,
        n_items,
        item_dim=64,
        hidden_dim=128,
        vocab_size=32,
        message_length=4,
    ):
        super().__init__()

        self.sender = Sender(
            n_items=n_items,
            item_dim=item_dim,
            hidden_dim=hidden_dim,
            vocab_size=vocab_size,
            message_length=message_length,
        )

        self.receiver = Receiver(
            n_items=n_items,
            item_dim=item_dim,
            hidden_dim=hidden_dim,
            vocab_size=vocab_size,
            message_length=message_length,
        )

    def forward(self, batch, temperature=1.0):
        sender_out = self.sender(
            history_items=batch["history_items"],
            history_events=batch["history_events"],
            history_deltas=batch["history_deltas"],
            temperature=temperature,
        )

        scores = self.receiver(
            message=sender_out["message"],
            candidates=batch["candidates"],
        )

        return {
            **sender_out,
            "scores": scores,
        }

In [56]:
model = LewisGame(
    n_items=N_ITEMS,
    item_dim=64,
    hidden_dim=128,
    vocab_size=32,
    message_length=4,
).to(DEVICE)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Parameters: {n_params:,}")

Parameters: 114,003,230


In [57]:
model.train()

batch_gpu = {
    k: v.to(DEVICE)
    for k, v in batch.items()
}

output = model(
    batch_gpu,
    temperature=1.0,
)

print("Scores:", output["scores"].shape)
print("Message:", output["message"].shape)
print("Message IDs:", output["message_ids"].shape)
print("User embedding:", output["user_embedding"].shape)

assert output["scores"].shape == (256, 4)
assert output["message"].shape == (256, 4, 32)
assert output["message_ids"].shape == (256, 4)

assert torch.allclose(
    output["message"].sum(dim=-1),
    torch.ones(
        256, 4,
        device=DEVICE,
    ),
)

loss = F.cross_entropy(
    output["scores"],
    batch_gpu["target"],
)

model.zero_grad(set_to_none=True)
loss.backward()

sender_grad = (
    model.sender.message_head.weight.grad
)

assert sender_grad is not None
assert torch.isfinite(sender_grad).all()
assert sender_grad.abs().sum().item() > 0

print("Loss:", loss.item())
print("Sender gradient norm:", sender_grad.norm().item())
print("Forward/backward checks passed!")

Scores: torch.Size([256, 4])
Message: torch.Size([256, 4, 32])
Message IDs: torch.Size([256, 4])
User embedding: torch.Size([256, 128])
Loss: 1.7066055536270142
Sender gradient norm: 0.0001895954046631232
Forward/backward checks passed!


In [58]:
from tqdm.auto import tqdm

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

def train_one_epoch(
    model,
    loader,
    optimizer,
    temperature=1.0,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in tqdm(loader, leave=False):

        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        output = model(
            batch,
            temperature=temperature,
        )

        loss = F.cross_entropy(
            output["scores"],
            batch["target"],
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        n = batch["target"].shape[0]

        total_loss += loss.item() * n

        predictions = (
            output["scores"].argmax(dim=1)
        )

        total_correct += (
            predictions == batch["target"]
        ).sum().item()

        total_count += n

    return {
        "loss": total_loss / total_count,
        "accuracy": total_correct / total_count,
    }

In [59]:
history = []

for epoch in range(1, 4):
    metrics = train_one_epoch(
        model=model,
        loader=smoke_loader,
        optimizer=optimizer,
        temperature=1.0,
    )

    history.append(metrics)

    print(
        f"Epoch {epoch}: "
        f"loss={metrics['loss']:.4f}, "
        f"accuracy={metrics['accuracy']:.4f}"
    )

Epoch 1: loss=1.7168, accuracy=0.2617


Epoch 2: loss=1.7072, accuracy=0.2664


Epoch 3: loss=1.6974, accuracy=0.2725


In [60]:
model.eval()

with torch.no_grad():
    output = model(batch_gpu)

messages = (
    output["message_ids"]
    .cpu()
    .numpy()
)

print("First 15 messages:")
print(messages[:15])

print(
    "Unique messages:",
    len(np.unique(messages, axis=0)),
    "out of",
    len(messages),
)

for position in range(messages.shape[1]):
    values, counts = np.unique(
        messages[:, position],
        return_counts=True,
    )

    print(
        f"Position {position}: "
        f"{len(values)} different symbols"
    )

First 15 messages:
[[13 23 14 23]
 [13  1 20 23]
 [13  2 14 23]
 [13 23 14 23]
 [13  1 14 23]
 [13  2 14 23]
 [13  1 14 23]
 [13  1 14 23]
 [13  1 14 23]
 [13  1 20 23]
 [13  1 14 23]
 [13  1 14 23]
 [13  1 14 23]
 [13  1 14 23]
 [13  1 14 23]]
Unique messages: 6 out of 256
Position 0: 1 different symbols
Position 1: 3 different symbols
Position 2: 2 different symbols
Position 3: 1 different symbols


### Receiver вообще использует сообщения?

In [62]:
@torch.no_grad()
def evaluate_communication(model, loader, device):
    model.eval()

    results = {
        "normal": [0, 0],
        "shuffled": [0, 0],
        "constant": [0, 0],
    }

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        out = model(batch)
        messages = out["message"]
        candidates = batch["candidates"]
        targets = batch["target"]

        B = targets.size(0)

        # 1. Настоящее сообщение
        normal_scores = out["scores"]

        # 2. Сообщения других пользователей
        permutation = torch.randperm(
            B, device=device
        )

        shuffled_scores = model.receiver(
            messages[permutation],
            candidates,
        )

        # 3. Одно сообщение для всех
        constant_messages = (
            messages[:1].expand_as(messages)
        )

        constant_scores = model.receiver(
            constant_messages,
            candidates,
        )

        for name, scores in [
            ("normal", normal_scores),
            ("shuffled", shuffled_scores),
            ("constant", constant_scores),
        ]:
            correct = (
                scores.argmax(dim=1) == targets
            ).sum().item()

            results[name][0] += correct
            results[name][1] += B

    return {
        name: correct / total
        for name, (correct, total) in results.items()
    }

In [63]:
diagnostics = evaluate_communication(
    model,
    smoke_loader,
    DEVICE,
)

for name, accuracy in diagnostics.items():
    print(f"{name}: {accuracy:.4f}")

normal: 0.2717
shuffled: 0.2715
constant: 0.2712


Энтропия сообщений

In [64]:
from scipy.stats import entropy

@torch.no_grad()
def message_statistics(model, loader, device):
    model.eval()

    all_messages = []

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        out = model(batch)

        all_messages.append(
            out["message_ids"].cpu().numpy()
        )

    messages = np.concatenate(
        all_messages, axis=0
    )

    print("Examples:", len(messages))

    print(
        "Unique messages:",
        len(np.unique(messages, axis=0)),
    )

    for j in range(messages.shape[1]):
        counts = np.bincount(
            messages[:, j],
            minlength=32,
        )

        probs = counts / counts.sum()

        h = entropy(probs, base=2)

        print(
            f"Position {j}: "
            f"entropy={h:.3f} bits, "
            f"symbols={(counts > 0).sum()}"
        )

    return messages

In [65]:
messages = message_statistics(
    model,
    smoke_loader,
    DEVICE,
)

Examples: 4096
Unique messages: 9
Position 0: entropy=0.016 bits, symbols=2
Position 1: entropy=1.077 bits, symbols=4
Position 2: entropy=0.533 bits, symbols=4
Position 3: entropy=0.000 bits, symbols=1


### Receiver без Sender

In [67]:
@torch.no_grad()
def evaluate_bias_only(model, loader, device):
    model.eval()

    n_correct = 0
    n_total = 0
    total_loss = 0.0

    for batch in loader:
        candidates = batch["candidates"].to(device)
        targets = batch["target"].to(device)

        scores = (
            model.receiver.item_bias(candidates)
            .squeeze(-1)
        )

        loss = F.cross_entropy(scores, targets)

        n = targets.shape[0]

        n_correct += (
            scores.argmax(dim=1) == targets
        ).sum().item()

        total_loss += loss.item() * n
        n_total += n

    return {
        "accuracy": n_correct / n_total,
        "loss": total_loss / n_total,
    }


bias_metrics = evaluate_bias_only(
    model,
    smoke_loader,
    DEVICE,
)

print(bias_metrics)

{'accuracy': 0.268310546875, 'loss': 1.7001673802733421}


Чувствительность оценок к сообщению

In [68]:
@torch.no_grad()
def measure_message_sensitivity(model, loader, device):
    model.eval()

    total_diff = 0.0
    total_score_std = 0.0
    total_changed = 0
    total_n = 0

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        out = model(batch)

        messages = out["message"]
        candidates = batch["candidates"]

        B = candidates.shape[0]

        shuffled = messages[
            torch.randperm(B, device=device)
        ]

        normal_scores = out["scores"]

        shuffled_scores = model.receiver(
            shuffled,
            candidates,
        )

        # Среднее изменение оценок
        total_diff += (
            (normal_scores - shuffled_scores)
            .abs().mean().item() * B
        )

        # Типичный разброс оценок внутри candidate set
        total_score_std += (
            normal_scores.std(dim=1)
            .mean().item() * B
        )

        # Доля примеров, где изменился победитель
        total_changed += (
            normal_scores.argmax(dim=1)
            != shuffled_scores.argmax(dim=1)
        ).sum().item()

        total_n += B

    return {
        "mean_absolute_score_change":
            total_diff / total_n,

        "mean_candidate_score_std":
            total_score_std / total_n,

        "prediction_changed_fraction":
            total_changed / total_n,
    }


sensitivity = measure_message_sensitivity(
    model,
    smoke_loader,
    DEVICE,
)

for key, value in sensitivity.items():
    print(f"{key}: {value:.6f}")

mean_absolute_score_change: 0.004075
mean_candidate_score_std: 0.921056
prediction_changed_fraction: 0.003174


Эх, получается, что Receiver практически не использует сообщения Sender.

### Непрерывная версия

Проверяем, а возможно ли вообще обучиться?

In [71]:
class ContinuousGame(nn.Module):

    def __init__(
        self,
        n_items,
        item_dim=64,
        hidden_dim=128,
    ):
        super().__init__()

        self.sender = Sender(
            n_items=n_items,
            item_dim=item_dim,
            hidden_dim=hidden_dim,
            vocab_size=32,
            message_length=4,
        )

        self.user_projection = nn.Sequential(
            nn.Linear(hidden_dim, item_dim),
            nn.LayerNorm(item_dim),
        )

        # Отдельной таблицы item embeddings нет.
        # Используем sender.item_embedding.

        self.logit_scale = nn.Parameter(
            torch.tensor(0.0)
        )

    def forward(self, batch):
        sender_out = self.sender(
            history_items=batch["history_items"],
            history_events=batch["history_events"],
            history_deltas=batch["history_deltas"],
        )

        user_embedding = self.user_projection(
            sender_out["user_embedding"]
        )

        candidate_embeddings = (
            self.sender.item_embedding(
                batch["candidates"]
            )
        )

        # Нормируем векторы
        user_embedding = F.normalize(
            user_embedding, dim=-1
        )

        candidate_embeddings = F.normalize(
            candidate_embeddings, dim=-1
        )

        scores = torch.einsum(
            "bd,bkd->bk",
            user_embedding,
            candidate_embeddings,
        )

        scale = self.logit_scale.exp().clamp(max=100.0)
        scores = scores * scale

        return {
            "scores": scores,
            "user_embedding": user_embedding,
        }

In [72]:
continuous_model = ContinuousGame(
    n_items=N_ITEMS,
    item_dim=64,
    hidden_dim=128,
).to(DEVICE)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in continuous_model.parameters()
    )
)

continuous_optimizer = torch.optim.Adam(
    (
        p for p in continuous_model.parameters()
        if p.requires_grad
    ),
    lr=1e-3,
)

Parameters: 56546786


In [73]:
def train_continuous_epoch(
    model,
    loader,
    optimizer,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for batch in loader:
        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        out = model(batch)

        loss = F.cross_entropy(
            out["scores"],
            batch["target"],
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0,
        )

        optimizer.step()

        n = len(batch["target"])
        total += n

        total_loss += loss.item() * n

        total_correct += (
            out["scores"].argmax(dim=1)
            == batch["target"]
        ).sum().item()

    return {
        "loss": total_loss / total,
        "accuracy": total_correct / total,
    }


for epoch in range(1, 11):
    metrics = train_continuous_epoch(
        continuous_model,
        smoke_loader,
        continuous_optimizer,
    )

    print(
        f"Epoch {epoch:02d}: "
        f"loss={metrics['loss']:.4f}, "
        f"accuracy={metrics['accuracy']:.4f}"
    )

Epoch 01: loss=1.3904, accuracy=0.2581
Epoch 02: loss=1.3762, accuracy=0.2896
Epoch 03: loss=1.3692, accuracy=0.3191
Epoch 04: loss=1.3627, accuracy=0.3345
Epoch 05: loss=1.3556, accuracy=0.3491
Epoch 06: loss=1.3466, accuracy=0.3762
Epoch 07: loss=1.3345, accuracy=0.4192
Epoch 08: loss=1.3176, accuracy=0.4700
Epoch 09: loss=1.2900, accuracy=0.5535
Epoch 10: loss=1.2561, accuracy=0.6438


In [74]:
@torch.no_grad()
def evaluate_continuous_communication(
    model,
    loader,
    device,
):
    model.eval()

    normal_correct = 0
    shuffled_correct = 0
    total = 0

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        out = model(batch)

        normal_scores = out["scores"]
        user_embeddings = out["user_embedding"]

        candidate_emb = F.normalize(
            model.sender.item_embedding(
                batch["candidates"]
            ),
            dim=-1,
        )

        B = user_embeddings.shape[0]

        shuffled_users = user_embeddings[
            torch.randperm(B, device=device)
        ]

        scale = model.logit_scale.exp().clamp(max=100.0)

        shuffled_scores = torch.einsum(
            "bd,bkd->bk",
            shuffled_users,
            candidate_emb,
        ) * scale

        targets = batch["target"]

        normal_correct += (
            normal_scores.argmax(dim=1)
            == targets
        ).sum().item()

        shuffled_correct += (
            shuffled_scores.argmax(dim=1)
            == targets
        ).sum().item()

        total += B

    return {
        "normal_accuracy": normal_correct / total,
        "shuffled_accuracy": shuffled_correct / total,
    }


result = evaluate_continuous_communication(
    continuous_model,
    smoke_loader,
    DEVICE,
)

print(result)

{'normal_accuracy': 0.73046875, 'shuffled_accuracy': 0.367919921875}


еее! архитектура способна передавать через Sender информацию, влияющую на решения Receiver.

In [75]:
from pathlib import Path

val_df = pl.read_parquet(
    str(GAME_DIR / "val" / "part_*.parquet")
)

val_dataset = LewisGameDataset(val_df)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

print("Validation examples:", len(val_dataset))

# Проверка доли неизвестных треков
val_candidates = val_dataset.candidates.numpy()

unk_fraction = (
    val_candidates == UNK_ID
).mean()

print(f"UNK candidates: {unk_fraction:.2%}")

Validation examples: 4213
UNK candidates: 0.59%


In [76]:
val_result = evaluate_continuous_communication(
    continuous_model,
    val_loader,
    DEVICE,
)

print("Validation:")
print(val_result)

Validation:
{'normal_accuracy': 0.258960360788037, 'shuffled_accuracy': 0.2475670543555661}


In [77]:
@torch.no_grad()
def evaluate_ranking_game(model, loader, device):
    model.eval()

    total = 0
    total_loss = 0.0
    total_correct = 0
    total_mrr = 0.0

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        scores = model(batch)["scores"]
        targets = batch["target"]

        n = targets.size(0)

        loss = F.cross_entropy(scores, targets)

        ranks = torch.argsort(
            scores, dim=1, descending=True
        )

        correct_rank = (
            ranks == targets[:, None]
        ).int().argmax(dim=1) + 1

        total_loss += loss.item() * n
        total_correct += (
            correct_rank == 1
        ).sum().item()

        total_mrr += (
            1.0 / correct_rank.float()
        ).sum().item()

        total += n

    return {
        "loss": total_loss / total,
        "accuracy@1": total_correct / total,
        "mrr@4": total_mrr / total,
    }


print(
    evaluate_ranking_game(
        continuous_model,
        val_loader,
        DEVICE,
    )
)

{'loss': 1.3900753732195044, 'accuracy@1': 0.258960360788037, 'mrr@4': 0.5298283203690006}


а нет, показалось :(

In [78]:
import numpy as np

def dataset_coverage(dataset, name):
    history = dataset.history_items.numpy()
    candidates = dataset.candidates.numpy()

    history_mask = history != PAD_ID

    print(f"\n{name}")

    print(
        "UNK in candidates:",
        round((candidates == UNK_ID).mean(), 4)
    )

    print(
        "UNK in non-PAD history:",
        round(
            (history[history_mask] == UNK_ID).mean(),
            4
        ) if history_mask.any() else None
    )

    print(
        "Rows with any UNK candidate:",
        round(
            (candidates == UNK_ID).any(axis=1).mean(),
            4
        )
    )


dataset_coverage(
    smoke_dataset,
    "Smoke train"
)

dataset_coverage(
    val_dataset,
    "Validation"
)


Smoke train
UNK in candidates: 0.0
UNK in non-PAD history: 0.0
Rows with any UNK candidate: 0.0

Validation
UNK in candidates: 0.0059
UNK in non-PAD history: 0.0076
Rows with any UNK candidate: 0.0237


In [79]:
import gc

from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

continuous_full = ContinuousGame(
    n_items=N_ITEMS,
    item_dim=64,
    hidden_dim=128,
).to(DEVICE)

optimizer_full = torch.optim.Adam(
    continuous_full.parameters(),
    lr=3e-4,
)

MAX_EPOCHS = 10
BATCH_SIZE = 1024
PATIENCE = 3

best_val_loss = float("inf")
epochs_without_improvement = 0

In [80]:
def train_full_epoch(
    model,
    optimizer,
    epoch,
    batch_size=1024,
):
    model.train()

    bucket_order = np.random.default_rng(
        SEED + epoch
    ).permutation(N_BUCKETS)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for bucket in tqdm(
        bucket_order,
        desc=f"Epoch {epoch}",
    ):
        path = (
            GAME_DIR
            / "train"
            / f"part_{bucket:03d}.parquet"
        )

        if not path.exists():
            continue

        df = pl.read_parquet(path)

        if df.is_empty():
            continue

        dataset = LewisGameDataset(df)

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
        )

        for batch in loader:
            batch = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            optimizer.zero_grad(set_to_none=True)

            output = model(batch)

            loss = F.cross_entropy(
                output["scores"],
                batch["target"],
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            n = batch["target"].size(0)

            total_loss += loss.item() * n

            total_correct += (
                output["scores"].argmax(dim=1)
                == batch["target"]
            ).sum().item()

            total_count += n

        del df, dataset, loader
        gc.collect()

    return {
        "loss": total_loss / total_count,
        "accuracy": total_correct / total_count,
        "n_examples": total_count,
    }

In [81]:
from pathlib import Path

CHECKPOINT_DIR = Path(
    "./lewis_game_data/checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

training_history = []

for epoch in range(1, MAX_EPOCHS + 1):

    train_metrics = train_full_epoch(
        model=continuous_full,
        optimizer=optimizer_full,
        epoch=epoch,
        batch_size=BATCH_SIZE,
    )

    val_metrics = evaluate_ranking_game(
        continuous_full,
        val_loader,
        DEVICE,
    )

    communication_metrics = (
        evaluate_continuous_communication(
            continuous_full,
            val_loader,
            DEVICE,
        )
    )

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy@1"],
        "val_mrr": val_metrics["mrr@4"],
        "val_shuffled_acc":
            communication_metrics["shuffled_accuracy"],
    }

    training_history.append(row)

    print(
        f"\nEpoch {epoch}: "
        f"train_loss={row['train_loss']:.4f}, "
        f"train_acc={row['train_acc']:.4f}, "
        f"val_loss={row['val_loss']:.4f}, "
        f"val_acc={row['val_acc']:.4f}, "
        f"val_mrr={row['val_mrr']:.4f}, "
        f"shuffled_acc={row['val_shuffled_acc']:.4f}"
    )

    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        epochs_without_improvement = 0

        torch.save(
            continuous_full.state_dict(),
            CHECKPOINT_DIR / "d2_full_best.pt",
        )

        print("Best checkpoint saved.")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

Epoch 1: 100%|██████████| 64/64 [01:08<00:00,  1.07s/it]



Epoch 1: train_loss=1.3527, train_acc=0.3392, val_loss=1.2985, val_acc=0.4272, val_mrr=0.6383, shuffled_acc=0.2827
Best checkpoint saved.


Epoch 2: 100%|██████████| 64/64 [01:07<00:00,  1.06s/it]



Epoch 2: train_loss=1.2636, train_acc=0.4628, val_loss=1.2463, val_acc=0.4662, val_mrr=0.6655, shuffled_acc=0.3022
Best checkpoint saved.


Epoch 3: 100%|██████████| 64/64 [01:08<00:00,  1.07s/it]


...


Epoch 8: train_loss=0.6006, train_acc=0.7819, val_loss=0.7891, val_acc=0.6929, val_mrr=0.8168, shuffled_acc=0.5891
Best checkpoint saved.


Epoch 9: 100%|██████████| 64/64 [01:08<00:00,  1.07s/it]



Epoch 9: train_loss=0.5227, train_acc=0.8087, val_loss=0.7605, val_acc=0.7099, val_mrr=0.8274, shuffled_acc=0.6048
Best checkpoint saved.


Epoch 10: 100%|██████████| 64/64 [01:08<00:00,  1.06s/it]



Epoch 10: train_loss=0.4627, train_acc=0.8311, val_loss=0.7459, val_acc=0.7237, val_mrr=0.8357, shuffled_acc=0.6373
Best checkpoint saved.


In [82]:
display(
    pl.DataFrame(training_history)
)

epoch,train_loss,train_acc,val_loss,val_acc,val_mrr,val_shuffled_acc
i64,f64,f64,f64,f64,f64,f64
1,1.352712,0.339176,1.298518,0.427249,0.638263,0.282696
2,1.263615,0.4628,1.246295,0.466176,0.665539,0.30216
3,1.189982,0.523493,1.183058,0.51436,0.699897,0.363399
4,1.091236,0.588471,1.099131,0.565868,0.733128,0.413482
5,0.964735,0.651823,0.999663,0.614764,0.765567,0.491099
6,0.827116,0.70569,0.914934,0.642298,0.783132,0.527178
7,0.701889,0.748363,0.835951,0.67268,0.803822,0.567054
8,0.600615,0.781882,0.789137,0.692855,0.816797,0.589129
9,0.522706,0.808707,0.760503,0.709945,0.82738,0.604795


Фух! Все-таки возможно обуучиться

In [83]:
def target_in_history_stats(lf, name):
    result = (
        lf.select(
            pl.len().alias("total"),

            (
                pl.col("history_item_ids")
                .list.contains(
                    pl.col("item_id").cast(pl.Int64) + 1
                )
            ).sum().alias("target_in_history"),
        )
        .with_columns(
            (
                pl.col("target_in_history")
                / pl.col("total")
            ).alias("fraction")
        )
        .collect()
    )

    print(f"\n{name}")
    display(result)


for name, lf in [
    ("train", game_train),
    ("val", game_val),
    ("test", game_test),
]:
    target_in_history_stats(lf, name)


train


total,target_in_history,fraction
u32,u32,f64
836152,23039,0.027554



val


total,target_in_history,fraction
u32,u32,f64
4213,90,0.021362



test


total,target_in_history,fraction
u32,u32,f64
3885,96,0.02471


In [84]:
@torch.no_grad()
def evaluate_seen_unseen(model, loader, device):
    model.eval()

    stats = {
        "seen": [0, 0],
        "unseen": [0, 0],
    }

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        scores = model(batch)["scores"]
        predictions = scores.argmax(dim=1)

        positive_items = batch["candidates"].gather(
            1,
            batch["target"].unsqueeze(1),
        )

        seen = (
            batch["history_items"] == positive_items
        ).any(dim=1)

        correct = predictions == batch["target"]

        for name, mask in [
            ("seen", seen),
            ("unseen", ~seen),
        ]:
            stats[name][0] += (
                correct & mask
            ).sum().item()

            stats[name][1] += mask.sum().item()

    return {
        name: {
            "n": total,
            "accuracy": correct / total if total else None,
        }
        for name, (correct, total) in stats.items()
    }


seen_results = evaluate_seen_unseen(
    continuous_full,
    val_loader,
    DEVICE,
)

print(seen_results)

{'seen': {'n': 1537, 'accuracy': 0.8952504879635654}, 'unseen': {'n': 2676, 'accuracy': 0.625186846038864}}


In [85]:
# Сырые ID: history_item_ids = original_id + 1
raw_history = np.asarray(
    val_df["history_item_ids"].to_list(),
    dtype=np.int64,
)

raw_positive = val_df["item_id"].to_numpy().astype(np.int64)

seen_raw = (
    raw_history == (raw_positive[:, None] + 1)
).any(axis=1)

# Уже индексированные ID
encoded_history = val_dataset.history_items.numpy()
encoded_candidates = val_dataset.candidates.numpy()
positions = val_dataset.targets.numpy()

encoded_positive = encoded_candidates[
    np.arange(len(positions)), positions
]

seen_encoded = (
    encoded_history == encoded_positive[:, None]
).any(axis=1)

print("Raw seen:", seen_raw.sum())
print("Encoded seen:", seen_encoded.sum())
print("Disagreements:", (seen_raw != seen_encoded).sum())

print(
    "Positive UNK:",
    (encoded_positive == UNK_ID).sum()
)

assert np.array_equal(
    seen_raw, seen_encoded
), "Raw/encoded seen indicators differ"

Raw seen: 1512
Encoded seen: 1537
Disagreements: 25
Positive UNK: 100


AssertionError: Raw/encoded seen indicators differ

### Дискретизируем

In [86]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

D2_PATH = CHECKPOINT_DIR / "d2_full_best.pt"

teacher = ContinuousGame(
    n_items=N_ITEMS,
    item_dim=64,
    hidden_dim=128,
).to(DEVICE)

teacher.load_state_dict(
    torch.load(
        D2_PATH,
        map_location=DEVICE,
        weights_only=True,
    )
)

teacher.eval()

for p in teacher.parameters():
    p.requires_grad_(False)

print("D2 teacher loaded")

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


D2 teacher loaded


In [90]:
class DiscreteChannel(nn.Module):
    def __init__(
        self,
        input_dim=128,
        item_dim=64,
        hidden_dim=128,
        vocab_size=32,
        message_length=4,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.message_length = message_length

        # Sender: непрерывный вектор -> logits символов
        self.sender_head = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                message_length * vocab_size,
            ),
        )

        # Receiver: дискретные символы -> векторы
        self.codebook = nn.Parameter(
            torch.randn(
                message_length,
                vocab_size,
                hidden_dim,
            ) * 0.02
        )

        self.decoder = nn.Sequential(
            nn.Linear(
                message_length * hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(hidden_dim, item_dim),
            nn.LayerNorm(item_dim),
        )

    def forward(self, h, temperature=1.0):
        B = h.size(0)

        logits = self.sender_head(h).reshape(
            B,
            self.message_length,
            self.vocab_size,
        )

        if self.training:
            message = F.gumbel_softmax(
                logits,
                tau=temperature,
                hard=True,
                dim=-1,
            )
        else:
            ids = logits.argmax(dim=-1)
            message = F.one_hot(
                ids,
                num_classes=self.vocab_size,
            ).float()

        symbols = torch.einsum(
            "blv,lvh->blh",
            message,
            self.codebook,
        )

        z = self.decoder(
            symbols.reshape(B, -1)
        )

        z = F.normalize(z, dim=-1)

        return {
            "message": message,
            "message_ids": message.argmax(dim=-1),
            "message_logits": logits,
            "decoded_embedding": z,
        }

In [91]:
class DiscreteGame(nn.Module):

    def __init__(
        self,
        pretrained_model,
        vocab_size=32,
        message_length=4,
        temperature=1.0,
    ):
        super().__init__()

        # Копируем предобученные компоненты.
        self.history_encoder = copy.deepcopy(
            pretrained_model.sender
        )

        self.teacher_projection = copy.deepcopy(
            pretrained_model.user_projection
        )

        self.channel = DiscreteChannel(
            input_dim=128,
            item_dim=64,
            hidden_dim=128,
            vocab_size=vocab_size,
            message_length=message_length,
        )

        self.logit_scale = nn.Parameter(
            pretrained_model.logit_scale.detach().clone()
        )

        self.temperature = temperature

        # На первом этапе замораживаем энкодер,
        # эмбеддинги и параметры учителя.
        for p in self.history_encoder.parameters():
            p.requires_grad_(False)

        for p in self.teacher_projection.parameters():
            p.requires_grad_(False)

        self.logit_scale.requires_grad_(False)

    def forward(self, batch):
        # История -> непрерывное представление
        with torch.no_grad():
            encoder_out = self.history_encoder(
                history_items=batch["history_items"],
                history_events=batch["history_events"],
                history_deltas=batch["history_deltas"],
            )

            h = encoder_out["user_embedding"]

            target_z = F.normalize(
                self.teacher_projection(h),
                dim=-1,
            )

        # Непрерывное представление -> дискретное сообщение
        out = self.channel(
            h,
            temperature=self.temperature,
        )

        z = out["decoded_embedding"]

        # Те же item embeddings, что в D2.
        candidate_emb = F.normalize(
            self.history_encoder.item_embedding(
                batch["candidates"]
            ),
            dim=-1,
        )

        scores = torch.einsum(
            "bd,bkd->bk",
            z,
            candidate_emb,
        )

        scores = scores * (
            self.logit_scale.exp().clamp(max=100.0)
        )

        return {
            **out,
            "scores": scores,
            "teacher_embedding": target_z,
        }

In [92]:
discrete_model = DiscreteGame(
    pretrained_model=teacher,
    vocab_size=32,
    message_length=4,
    temperature=1.0,
).to(DEVICE)

optimizer_d3 = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        discrete_model.parameters(),
    ),
    lr=1e-3,
    weight_decay=1e-4,
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in discrete_model.parameters()
        if p.requires_grad
    )
)

Trainable parameters: 123712


In [93]:
discrete_model.train()

batch_gpu = {
    k: v.to(DEVICE)
    for k, v in next(iter(smoke_loader)).items()
}

out = discrete_model(batch_gpu)

ce_loss = F.cross_entropy(
    out["scores"],
    batch_gpu["target"],
)

distill_loss = (
    1 - F.cosine_similarity(
        out["decoded_embedding"],
        out["teacher_embedding"],
        dim=-1,
    )
).mean()

loss = ce_loss + 0.5 * distill_loss

optimizer_d3.zero_grad(set_to_none=True)
loss.backward()

grad = (
    discrete_model.channel
    .sender_head[-1].weight.grad
)

print("CE loss:", ce_loss.item())
print("Distillation loss:", distill_loss.item())
print("Total loss:", loss.item())
print("Sender gradient norm:", grad.norm().item())

assert out["scores"].shape == (256, 4)
assert out["message_ids"].shape == (256, 4)
assert torch.isfinite(loss)
assert grad is not None
assert torch.isfinite(grad).all()
assert grad.abs().sum().item() > 0

print("D3 forward/backward checks passed!")

CE loss: 2.5934417247772217
Distillation loss: 1.1946113109588623
Total loss: 3.1907472610473633
Sender gradient norm: 0.024663617834448814
D3 forward/backward checks passed!


In [94]:
from tqdm.auto import tqdm
import gc

def train_discrete_epoch(
    model,
    optimizer,
    epoch,
    lambda_distill=0.5,
    batch_size=1024,
):
    model.train()

    total_loss = 0.0
    total_ce = 0.0
    total_distill = 0.0
    total_correct = 0
    total = 0

    bucket_order = np.random.default_rng(
        SEED + 1000 + epoch
    ).permutation(N_BUCKETS)

    for bucket in tqdm(
        bucket_order,
        desc=f"D3 epoch {epoch}",
    ):
        path = (
            GAME_DIR
            / "train"
            / f"part_{bucket:03d}.parquet"
        )

        if not path.exists():
            continue

        df = pl.read_parquet(path)
        dataset = LewisGameDataset(df)

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
        )

        for batch in loader:
            batch = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            optimizer.zero_grad(set_to_none=True)

            out = model(batch)

            ce = F.cross_entropy(
                out["scores"],
                batch["target"],
            )

            distill = (
                1 - F.cosine_similarity(
                    out["decoded_embedding"],
                    out["teacher_embedding"],
                    dim=-1,
                )
            ).mean()

            loss = ce + lambda_distill * distill

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            n = len(batch["target"])

            total += n
            total_loss += loss.item() * n
            total_ce += ce.item() * n
            total_distill += distill.item() * n

            total_correct += (
                out["scores"].argmax(dim=1)
                == batch["target"]
            ).sum().item()

        del df, dataset, loader
        gc.collect()

    return {
        "loss": total_loss / total,
        "ce_loss": total_ce / total,
        "distill_loss": total_distill / total,
        "accuracy": total_correct / total,
    }

In [95]:
from tqdm.auto import tqdm
import gc

def train_discrete_epoch(
    model,
    optimizer,
    epoch,
    lambda_distill=0.5,
    batch_size=1024,
):
    model.train()

    total_loss = 0.0
    total_ce = 0.0
    total_distill = 0.0
    total_correct = 0
    total = 0

    bucket_order = np.random.default_rng(
        SEED + 1000 + epoch
    ).permutation(N_BUCKETS)

    for bucket in tqdm(
        bucket_order,
        desc=f"D3 epoch {epoch}",
    ):
        path = (
            GAME_DIR
            / "train"
            / f"part_{bucket:03d}.parquet"
        )

        if not path.exists():
            continue

        df = pl.read_parquet(path)
        dataset = LewisGameDataset(df)

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
        )

        for batch in loader:
            batch = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            optimizer.zero_grad(set_to_none=True)

            out = model(batch)

            ce = F.cross_entropy(
                out["scores"],
                batch["target"],
            )

            distill = (
                1 - F.cosine_similarity(
                    out["decoded_embedding"],
                    out["teacher_embedding"],
                    dim=-1,
                )
            ).mean()

            loss = ce + lambda_distill * distill

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            n = len(batch["target"])

            total += n
            total_loss += loss.item() * n
            total_ce += ce.item() * n
            total_distill += distill.item() * n

            total_correct += (
                out["scores"].argmax(dim=1)
                == batch["target"]
            ).sum().item()

        del df, dataset, loader
        gc.collect()

    return {
        "loss": total_loss / total,
        "ce_loss": total_ce / total,
        "distill_loss": total_distill / total,
        "accuracy": total_correct / total,
    }

In [96]:
D3_EPOCHS = 3

d3_history = []
best_d3_val_loss = float("inf")

for epoch in range(1, D3_EPOCHS + 1):

    train_metrics = train_discrete_epoch(
        model=discrete_model,
        optimizer=optimizer_d3,
        epoch=epoch,
        lambda_distill=0.5,
        batch_size=1024,
    )

    val_metrics = evaluate_ranking_game(
        discrete_model,
        val_loader,
        DEVICE,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_ce": train_metrics["ce_loss"],
        "train_distill": train_metrics["distill_loss"],
        "train_acc": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy@1"],
        "val_mrr": val_metrics["mrr@4"],
    }

    d3_history.append(row)

    print(
        f"\nEpoch {epoch}: "
        f"train_acc={row['train_acc']:.4f}, "
        f"val_acc={row['val_acc']:.4f}, "
        f"val_mrr={row['val_mrr']:.4f}, "
        f"val_loss={row['val_loss']:.4f}"
    )

    if row["val_loss"] < best_d3_val_loss:
        best_d3_val_loss = row["val_loss"]

        torch.save(
            discrete_model.state_dict(),
            CHECKPOINT_DIR / "d3b_frozen_best.pt",
        )

        print("Best D3 checkpoint saved.")

D3 epoch 1: 100%|██████████| 64/64 [00:44<00:00,  1.42it/s]



Epoch 1: train_acc=0.7474, val_acc=0.6912, val_mrr=0.8167, val_loss=0.8405
Best D3 checkpoint saved.


D3 epoch 2: 100%|██████████| 64/64 [00:44<00:00,  1.43it/s]



Epoch 2: train_acc=0.7626, val_acc=0.6826, val_mrr=0.8137, val_loss=0.8381
Best D3 checkpoint saved.


D3 epoch 3: 100%|██████████| 64/64 [00:44<00:00,  1.42it/s]



Epoch 3: train_acc=0.7661, val_acc=0.6862, val_mrr=0.8157, val_loss=0.8367
Best D3 checkpoint saved.


In [97]:
display(pl.DataFrame(d3_history))

epoch,train_loss,train_ce,train_distill,train_acc,val_loss,val_acc,val_mrr
i64,f64,f64,f64,f64,f64,f64,f64
1,0.753953,0.671504,0.164899,0.747419,0.840468,0.691194,0.816679
2,0.699931,0.626363,0.147137,0.762619,0.838076,0.682649,0.813731
3,0.689216,0.61703,0.144371,0.766065,0.836701,0.686209,0.815749


### Изучаем язык

In [98]:
d3_messages = message_statistics(
    discrete_model,
    val_loader,
    DEVICE,
)

Examples: 4213
Unique messages: 680
Position 0: entropy=2.748 bits, symbols=9
Position 1: entropy=1.807 bits, symbols=7
Position 2: entropy=2.460 bits, symbols=8
Position 3: entropy=2.171 bits, symbols=8


In [101]:
discrete_model.load_state_dict(
    torch.load(
        CHECKPOINT_DIR / "d3b_frozen_best.pt",
        map_location=DEVICE,
        weights_only=True,
    )
)

discrete_model.eval()

print("Best checkpoint loaded")

communication_d3 = evaluate_d3_communication(
    discrete_model,
    val_loader,
    DEVICE,
)

for key, value in communication_d3.items():
    print(f"{key}: {value:.6f}")

Best checkpoint loaded
normal: 0.686209
shuffled: 0.668170
constant: 0.667221
mean_score_change: 0.500954
prediction_changed_fraction: 0.146451


In [100]:
@torch.no_grad()
def evaluate_d3_communication(model, loader, device):
    model.eval()

    results = {
        "normal": [0, 0],
        "shuffled": [0, 0],
        "constant": [0, 0],
    }

    score_diffs = []
    changed_predictions = []

    def scores_from_messages(messages, candidates):
        # Декодирование произвольных сообщений Receiver
        codebook = model.channel.codebook

        symbols = torch.einsum(
            "blv,lvh->blh",
            messages,
            codebook,
        )

        B = messages.shape[0]

        z = model.channel.decoder(
            symbols.reshape(B, -1)
        )

        z = F.normalize(z, dim=-1)

        candidate_emb = F.normalize(
            model.history_encoder.item_embedding(
                candidates
            ),
            dim=-1,
        )

        scale = model.logit_scale.exp().clamp(max=100.0)

        return torch.einsum(
            "bd,bkd->bk",
            z,
            candidate_emb,
        ) * scale

    for batch in loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        out = model(batch)

        messages = out["message"]
        candidates = batch["candidates"]
        targets = batch["target"]

        B = targets.shape[0]

        normal_scores = out["scores"]

        perm = torch.randperm(B, device=device)

        shuffled_scores = scores_from_messages(
            messages[perm],
            candidates,
        )

        constant_scores = scores_from_messages(
            messages[:1].expand_as(messages),
            candidates,
        )

        for name, scores in [
            ("normal", normal_scores),
            ("shuffled", shuffled_scores),
            ("constant", constant_scores),
        ]:
            correct = (
                scores.argmax(dim=1) == targets
            ).sum().item()

            results[name][0] += correct
            results[name][1] += B

        score_diffs.append(
            (normal_scores - shuffled_scores)
            .abs()
            .mean(dim=1)
            .cpu()
        )

        changed_predictions.append(
            (
                normal_scores.argmax(dim=1)
                != shuffled_scores.argmax(dim=1)
            ).cpu()
        )

    metrics = {
        name: correct / total
        for name, (correct, total) in results.items()
    }

    metrics["mean_score_change"] = (
        torch.cat(score_diffs).float().mean().item()
    )

    metrics["prediction_changed_fraction"] = (
        torch.cat(changed_predictions)
        .float().mean().item()
    )

    return metrics

In [103]:
discrete_model.load_state_dict(
    torch.load(
        CHECKPOINT_DIR / "d3b_frozen_best.pt",
        map_location=DEVICE,
        weights_only=True,
    )
)

discrete_model.eval()

print("Best checkpoint loaded")

communication_d3 = evaluate_d3_communication(
    discrete_model,
    val_loader,
    DEVICE,
)

for key, value in communication_d3.items():
    print(f"{key}: {value:.6f}")

Best checkpoint loaded
normal: 0.686209
shuffled: 0.661999
constant: 0.667221
mean_score_change: 0.504329
prediction_changed_fraction: 0.145977


In [104]:
raw_seen = (
    raw_history == (raw_positive[:, None] + 1)
).any(axis=1)

encoded_seen = (
    encoded_history == encoded_positive[:, None]
).any(axis=1)

mismatch = np.flatnonzero(raw_seen != encoded_seen)

print("Raw seen:", raw_seen.sum())
print("Encoded seen:", encoded_seen.sum())
print("Mismatches:", len(mismatch))

print(
    "Mismatches with positive UNK:",
    (encoded_positive[mismatch] == UNK_ID).sum()
)

for idx in mismatch[:5]:
    print(
        "row:", idx,
        "raw_positive:", raw_positive[idx],
        "encoded_positive:", encoded_positive[idx],
        "history_has_UNK:",
        (encoded_history[idx] == UNK_ID).any(),
        "raw_seen:", raw_seen[idx],
        "encoded_seen:", encoded_seen[idx],
    )

Raw seen: 1512
Encoded seen: 1537
Mismatches: 25
Mismatches with positive UNK: 25
row: 6 raw_positive: 8805497 encoded_positive: 1 history_has_UNK: True raw_seen: False encoded_seen: True
row: 7 raw_positive: 5707109 encoded_positive: 1 history_has_UNK: True raw_seen: False encoded_seen: True
row: 260 raw_positive: 1953302 encoded_positive: 1 history_has_UNK: True raw_seen: False encoded_seen: True
row: 267 raw_positive: 4027103 encoded_positive: 1 history_has_UNK: True raw_seen: False encoded_seen: True
row: 441 raw_positive: 1232336 encoded_positive: 1 history_has_UNK: True raw_seen: False encoded_seen: True


In [ ]:
! pip freeze > requirements_03.txt